In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TestSparkLocal") \
    .config("spark.driver.memory", "6g") \
    .getOrCreate()

print("Spark version:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/10 22:33:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1


In [3]:
csv_path = "../us_accidents_cleaned2/cleanedUS.csv" 
df = spark.read.option("header", True).option("inferSchema", True).csv(csv_path)

df = spark.read.csv(csv_path, header=True, inferSchema=True)
print("Righe:", df.count())
df.show(5, truncate=False)


Righe: 7485236
+---------------+-----+-----+-----------+-----------------+---------+-------+--------+-------------------+-------------------+-----------------+-------------------+------------+----------------------------------------------------------------------------------------------------------------------------------+---------------+-----------+----------+--------------+--------------+--------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+----+---+-------+----+---------------+
|weather_grouped|State|Month|Humidity(%)|Precipitation(in)|ID       |Source |Severity|Start_Time         |End_Time           |Start_Lat        |Start_Lng          |Distance(mi)|Description                                                                                                                       |City           |County     |Timezone  |Temperature(F)|Visibility(mi)|Wind_Direction|Amenity|Bump |Crossin

26/01/10 22:34:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [5]:
df.columns

['weather_grouped',
 'State',
 'Month',
 'Humidity(%)',
 'Precipitation(in)',
 'ID',
 'Source',
 'Severity',
 'Start_Time',
 'End_Time',
 'Start_Lat',
 'Start_Lng',
 'Distance(mi)',
 'Description',
 'City',
 'County',
 'Timezone',
 'Temperature(F)',
 'Visibility(mi)',
 'Wind_Direction',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way',
 'Junction',
 'No_Exit',
 'Railway',
 'Roundabout',
 'Station',
 'Stop',
 'Traffic_Calming',
 'Traffic_Signal',
 'Turning_Loop',
 'Sunrise_Sunset',
 'Year',
 'Day',
 'Weekday',
 'Hour',
 'Wind_Speed(mph)']

In [7]:
import pandas as pd

# Seleziona solo colonne numeriche
num_cols = [c for c, t in df.dtypes if t in ("double", "int")]

pdf = df.select(num_cols).sample(fraction=0.4, seed=42).toPandas()

# Calcola la correlation matrix
corr = pdf.corr(method='pearson')

# Mostra la mappa di correlazione con valori numerici
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print(" Correlation matrix (Pearson):")
print(corr.round(2))

🔹 Correlation matrix (Pearson):
                   Month  Humidity(%)  Precipitation(in)  Severity  Start_Lat  \
Month               1.00         0.04              -0.00     -0.01       0.00   
Humidity(%)         0.04         1.00               0.16      0.02       0.02   
Precipitation(in)  -0.00         0.16               1.00      0.02      -0.01   
Severity           -0.01         0.02               0.02      1.00       0.07   
Start_Lat           0.00         0.02              -0.01      0.07       1.00   
Start_Lng           0.01         0.18               0.04      0.05      -0.07   
Distance(mi)        0.00         0.01               0.01      0.04       0.06   
Temperature(F)      0.14        -0.33              -0.01     -0.02      -0.45   
Visibility(mi)      0.02        -0.42              -0.26     -0.01      -0.11   
Sunrise_Sunset      0.05         0.26              -0.01      0.01       0.04   
Year               -0.16        -0.03              -0.01     -0.25      -0.04

## Creazione duration_minutes

In [5]:
from pyspark.sql.functions import unix_timestamp, col

df = df.withColumn(
    "duration_minutes",
    (unix_timestamp("End_Time") - unix_timestamp("Start_Time")) / 60
)

# Filtriamo valori impossibili
df = df.filter(col("duration_minutes") > 0)
df = df.filter(col("duration_minutes") < 1440)  # max 1 giorno


## Correlazione duration con le altre

In [8]:
import pandas as pd
num_cols = [c for c, t in df.dtypes if t in ("double", "int")]

if "duration_minutes" not in num_cols:
    num_cols.append("duration_minutes")

# ============================
# Sample + conversione in Pandas
# ============================

pdf = (
    df
    .select(num_cols)
    .dropna()
    .sample(fraction=0.4, seed=42)
    .toPandas()
)

# ============================
#  Correlation matrix (Pearson)
# ============================

corr = pdf.corr(method="pearson")

# ============================
# Output leggibile
# ============================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("Correlation matrix (Pearson):")
print(corr.round(2))

Correlation matrix (Pearson):
                   Month  Humidity(%)  Precipitation(in)  Severity  Start_Lat  \
Month               1.00         0.04              -0.00     -0.01       0.00   
Humidity(%)         0.04         1.00               0.16      0.02       0.02   
Precipitation(in)  -0.00         0.16               1.00      0.02      -0.01   
Severity           -0.01         0.02               0.02      1.00       0.07   
Start_Lat           0.00         0.02              -0.01      0.07       1.00   
Start_Lng           0.01         0.18               0.04      0.05      -0.07   
Distance(mi)        0.00         0.01               0.01      0.03       0.06   
Temperature(F)      0.14        -0.33              -0.01     -0.02      -0.45   
Visibility(mi)      0.03        -0.42              -0.27     -0.01      -0.11   
Sunrise_Sunset      0.05         0.26              -0.00      0.01       0.04   
Year               -0.16        -0.03              -0.01     -0.25      -0.04  

In [5]:
df_sample = df.sample(withReplacement=False, fraction=0.4, seed=42)
print("Sample size:", df_sample.count())


Sample size: 2984091


In [6]:
num_cols = [
    "Temperature(F)", "Humidity(%)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)",
    "Severity", "Distance(mi)"
]

target = "duration_minutes"

df_reg = df_sample.select(num_cols + [target]).dropna()


In [7]:
train_df, test_df = df_reg.randomSplit([0.7, 0.3], seed=42)


In [17]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=num_cols,
    outputCol="features"
)

dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol=target
)

pipeline = Pipeline(stages=[assembler, dt])


In [19]:
model = pipeline.fit(train_df)


In [20]:
pred = model.transform(test_df)
pred.select("prediction", target, "features").show(5)


+------------------+------------------+--------------------+
|        prediction|  duration_minutes|            features|
+------------------+------------------+--------------------+
|147.92297696087667|120.81666666666666|[-21.0,63.0,10.0,...|
| 136.1638374227478| 78.03333333333333|[-19.0,70.0,10.0,...|
|211.86385885294857|165.26666666666668|[-19.0,79.0,10.0,...|
| 136.1638374227478|29.633333333333333|[-18.0,67.0,10.0,...|
| 136.1638374227478|             81.55|[-17.0,76.0,10.0,...|
+------------------+------------------+--------------------+
only showing top 5 rows


In [21]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse = RegressionEvaluator(
    labelCol=target,
    predictionCol="prediction",
    metricName="rmse"
).evaluate(pred)

r2 = RegressionEvaluator(
    labelCol=target,
    predictionCol="prediction",
    metricName="r2"
).evaluate(pred)

print("RMSE:", rmse)
print("R2:", r2)


RMSE: 111.16946044788718
R2: 0.1769152919917052


## BASE MODEL - 4 FEATURE

In [23]:
from pyspark.sql.functions import unix_timestamp, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# ==========================================================
# CREAZIONE TARGET
# ==========================================================
df = df.withColumn(
    "duration_minutes",
    (unix_timestamp("End_Time") - unix_timestamp("Start_Time")) / 60
)
df = df.filter((col("duration_minutes") > 0) & (col("duration_minutes") < 1440))

# ==========================================================
# SAMPLE (40%)
# ==========================================================
df_sample = df.sample(withReplacement=False, fraction=0.4, seed=42)

# ==========================================================
# FEATURE SELECTION (BASE MODEL)
# ==========================================================
num_cols_base = ["Distance(mi)", "Severity", "Hour", "Month"]

df_reg = df_sample.select(num_cols_base + ["duration_minutes"]).dropna()

# ==========================================================
# TRAIN/TEST SPLIT
# ==========================================================
train_df, test_df = df_reg.randomSplit([0.7, 0.3], seed=42)

# ==========================================================
# ASSEMBLER + MODELLO
# ==========================================================
assembler = VectorAssembler(inputCols=num_cols_base, outputCol="features")
dt = DecisionTreeRegressor(featuresCol="features", labelCol="duration_minutes")

pipeline = Pipeline(stages=[assembler, dt])

# ==========================================================
# FIT
# ==========================================================
model = pipeline.fit(train_df)

# ==========================================================
# PREDICT
# ==========================================================
pred = model.transform(test_df)

# ==========================================================
# EVALUATION
# ==========================================================
from pyspark.ml.evaluation import RegressionEvaluator

# RMSE
rmse = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="rmse"
).evaluate(pred)

# MSE
mse = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="mse"
).evaluate(pred)

# MAE
mae = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="mae"
).evaluate(pred)

# R2
r2 = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="r2"
).evaluate(pred)

print("RMSE:", rmse)
print("MSE :", mse)
print("MAE :", mae)
print("R2  :", r2)



RMSE: 111.48206613970585
MSE : 12428.251070777751
MAE : 64.48662289403678
R2  : 0.17358356920074425


## WEATHER MODEL - 8 feature

In [25]:
from pyspark.sql.functions import unix_timestamp, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# ==========================================================
# CREAZIONE TARGET
# ==========================================================
df = df.withColumn(
    "duration_minutes",
    (unix_timestamp("End_Time") - unix_timestamp("Start_Time")) / 60
)
df = df.filter((col("duration_minutes") > 0) & (col("duration_minutes") < 1440))

# ==========================================================
# SAMPLE 40%
# ==========================================================
df_sample = df.sample(withReplacement=False, fraction=0.4, seed=42)

# ==========================================================
# FEATURE SET: WEATHER MODEL
# ==========================================================
num_cols_weather = [
    "Distance(mi)", "Severity", "Hour", "Month",
    "Temperature(F)", "Visibility(mi)",
    "Precipitation(in)", "Wind_Speed(mph)"
]

df_reg = df_sample.select(num_cols_weather + ["duration_minutes"]).dropna()

# ==========================================================
# TRAIN/TEST SPLIT
# ==========================================================
train_df, test_df = df_reg.randomSplit([0.7, 0.3], seed=42)

# ==========================================================
# ASSEMBLER + MODELLO
# ==========================================================
assembler = VectorAssembler(inputCols=num_cols_weather, outputCol="features")
dt = DecisionTreeRegressor(featuresCol="features", labelCol="duration_minutes")

pipeline = Pipeline(stages=[assembler, dt])

# ==========================================================
# FIT
# ==========================================================
model = pipeline.fit(train_df)

# ==========================================================
# PREDICT
# ==========================================================
pred = model.transform(test_df)

# ==========================================================
# EVALUATION
# ==========================================================
# RMSE
rmse = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="rmse"
).evaluate(pred)

# MSE
mse = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="mse"
).evaluate(pred)

# MAE
mae = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="mae"
).evaluate(pred)

# R2
r2 = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="r2"
).evaluate(pred)

print("RMSE:", rmse)
print("MSE :", mse)
print("MAE :", mae)
print("R2  :", r2)

RMSE: 110.95722406125606
MSE : 12311.50557137978
MAE : 62.47883684544853
R2  : 0.1836365410010058


## FULL MODEL- 13 feature

In [27]:
from pyspark.sql.functions import unix_timestamp, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

# ==========================================================
# TARGET
# ==========================================================
df = df.withColumn(
    "duration_minutes",
    (unix_timestamp("End_Time") - unix_timestamp("Start_Time")) / 60
)
df = df.filter((col("duration_minutes") > 0) & (col("duration_minutes") < 1440))

# ==========================================================
# SAMPLE 40%
# ==========================================================
df_sample = df.sample(withReplacement=False, fraction=0.4, seed=42)

# ==========================================================
# FEATURE SET: FULL MODEL
# ==========================================================
num_cols_full = [
    "Distance(mi)", "Severity",
    "Hour", "Weekday", "Month", "Year",
    "Temperature(F)", "Visibility(mi)", "Precipitation(in)", "Wind_Speed(mph)",
    "Humidity(%)",
    "Start_Lat", "Start_Lng"
]

df_reg = df_sample.select(num_cols_full + ["duration_minutes"]).dropna()

# ==========================================================
# TRAIN/TEST SPLIT
# ==========================================================
train_df, test_df = df_reg.randomSplit([0.7, 0.3], seed=42)

# ==========================================================
# ASSEMBLER + MODELLO
# ==========================================================
assembler = VectorAssembler(inputCols=num_cols_full, outputCol="features")
dt = DecisionTreeRegressor(featuresCol="features", labelCol="duration_minutes")

pipeline = Pipeline(stages=[assembler, dt])

# ==========================================================
# FIT
# ==========================================================
model = pipeline.fit(train_df)

# ==========================================================
# PREDICT
# ==========================================================
pred = model.transform(test_df)

# ==========================================================
# EVALUATION
# ==========================================================
# RMSE
rmse = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="rmse"
).evaluate(pred)

# MSE
mse = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="mse"
).evaluate(pred)

# MAE
mae = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="mae"
).evaluate(pred)

# R2
r2 = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="r2"
).evaluate(pred)

print("RMSE:", rmse)
print("MSE :", mse)
print("MAE :", mae)
print("R2  :", r2)

RMSE: 103.36304673116622
MSE : 10683.919429549253
MAE : 54.522696866105164
R2  : 0.2885017294566077


## BASE MODEL (Ottimizzato)

In [29]:
from pyspark.sql.functions import unix_timestamp, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# ===== TARGET =====
df = df.withColumn(
    "duration_minutes",
    (unix_timestamp("End_Time") - unix_timestamp("Start_Time")) / 60
)
df = df.filter((col("duration_minutes") > 0) & (col("duration_minutes") < 1440))

# ===== SAMPLE =====
df_sample = df.sample(withReplacement=False, fraction=0.4, seed=42)

# ===== FEATURES BASE =====
num_cols_base = ["Distance(mi)", "Severity", "Hour", "Month"]
df_reg = df_sample.select(num_cols_base + ["duration_minutes"]).dropna()

# ===== SPLIT =====
train_df, test_df = df_reg.randomSplit([0.7, 0.3], seed=42)

# ===== PIPELINE =====
assembler = VectorAssembler(inputCols=num_cols_base, outputCol="features")
dt = DecisionTreeRegressor(labelCol="duration_minutes", featuresCol="features")

pipeline = Pipeline(stages=[assembler, dt])

# ===== GRID FOR TUNING =====
paramGrid = (ParamGridBuilder()
             .addGrid(dt.maxDepth, [5, 10, 15])
             .addGrid(dt.maxBins, [32, 64])
             .addGrid(dt.minInstancesPerNode, [1, 5])
             .addGrid(dt.minInfoGain, [0.0, 0.05])
             .build())

# ===== TRAIN VALIDATION SPLIT =====
evaluator = RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName="rmse")

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    trainRatio=0.8
)

# ===== FIT =====
best_model_base = tvs.fit(train_df)

# ===== PREDICT =====
pred = best_model_base.transform(test_df)

# ===== METRICS =====
evals = {}
for m in ["rmse", "mse", "mae", "r2"]:
    evals[m] = RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName=m).evaluate(pred)

print("\n=== BASE MODEL OPTIMIZED ===")
print(evals)


26/01/06 17:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1323.2 KiB
26/01/06 17:51:14 WARN DAGScheduler: Broadcasting large task binary with size 1918.2 KiB
26/01/06 17:51:15 WARN DAGScheduler: Broadcasting large task binary with size 1461.3 KiB
26/01/06 17:51:15 WARN DAGScheduler: Broadcasting large task binary with size 1462.4 KiB
26/01/06 17:51:20 WARN DAGScheduler: Broadcasting large task binary with size 1303.9 KiB
26/01/06 17:51:21 WARN DAGScheduler: Broadcasting large task binary with size 1888.2 KiB
26/01/06 17:51:22 WARN DAGScheduler: Broadcasting large task binary with size 1438.3 KiB
26/01/06 17:51:22 WARN DAGScheduler: Broadcasting large task binary with size 1439.4 KiB
26/01/06 17:51:27 WARN DAGScheduler: Broadcasting large task binary with size 1178.2 KiB
26/01/06 17:51:28 WARN DAGScheduler: Broadcasting large task binary with size 1674.2 KiB
26/01/06 17:51:29 WARN DAGScheduler: Broadcasting large task binary with size 1255.2 KiB
26/01/06 17:51:29 WAR


=== BASE MODEL OPTIMIZED ===
{'rmse': 108.41647284655356, 'mse': 11754.131584487486, 'mae': 62.644284747552, 'r2': 0.2184091377074937}


26/01/06 17:53:22 WARN DAGScheduler: Broadcasting large task binary with size 1203.8 KiB


In [30]:
best_pipe = best_model_base.bestModel
best_dt = best_pipe.stages[1]

print("\n=== BEST HYPERPARAMETERS (DecisionTreeRegressor) ===")
print("maxDepth:", best_dt.getMaxDepth())
print("maxBins:", best_dt.getMaxBins())
print("minInstancesPerNode:", best_dt.getMinInstancesPerNode())
print("minInfoGain:", best_dt.getMinInfoGain())


=== BEST HYPERPARAMETERS (DecisionTreeRegressor) ===
maxDepth: 15
maxBins: 32
minInstancesPerNode: 5
minInfoGain: 0.05


## WEATHER MODEL (Ottimizzato)

In [32]:
from pyspark.sql.functions import unix_timestamp, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

num_cols_weather = [
    "Distance(mi)", "Severity", "Hour", "Month",
    "Temperature(F)", "Visibility(mi)", "Precipitation(in)", "Wind_Speed(mph)"
]

df_reg = df_sample.select(num_cols_weather + ["duration_minutes"]).dropna()

train_df, test_df = df_reg.randomSplit([0.7, 0.3], seed=42)

assembler = VectorAssembler(inputCols=num_cols_weather, outputCol="features")
dt = DecisionTreeRegressor(labelCol="duration_minutes", featuresCol="features")

pipeline = Pipeline(stages=[assembler, dt])

paramGrid = (ParamGridBuilder()
             .addGrid(dt.maxDepth, [5, 10, 20])
             .addGrid(dt.maxBins, [32, 64])
             .addGrid(dt.minInstancesPerNode, [1, 3])
             .build())

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName="rmse"),
    trainRatio=0.8
)

best_model_weather = tvs.fit(train_df)
pred = best_model_weather.transform(test_df)

evals = {}
for m in ["rmse", "mse", "mae", "r2"]:
    evals[m] = RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName=m).evaluate(pred)

print("\n=== WEATHER MODEL OPTIMIZED ===")
print(evals)


26/01/06 17:54:55 WARN DAGScheduler: Broadcasting large task binary with size 1656.5 KiB
26/01/06 17:54:56 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/01/06 17:54:59 WARN DAGScheduler: Broadcasting large task binary with size 4.3 MiB
26/01/06 17:55:03 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
26/01/06 17:55:07 WARN DAGScheduler: Broadcasting large task binary with size 1479.1 KiB
26/01/06 17:55:09 WARN DAGScheduler: Broadcasting large task binary with size 10.4 MiB
26/01/06 17:55:13 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/01/06 17:55:16 WARN DAGScheduler: Broadcasting large task binary with size 15.6 MiB
26/01/06 17:55:23 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/01/06 17:55:26 WARN DAGScheduler: Broadcasting large task binary with size 22.7 MiB
26/01/06 17:55:36 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/01/06 17:55:41 WARN DAGScheduler: Broadcas

[1191,417s][warning][gc,alloc] Executor task launch worker for task 12.0 in stage 1063.0 (TID 18946): Retried waiting for GCLocker too often allocating 111590 words


26/01/06 17:57:13 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/01/06 17:57:18 WARN DAGScheduler: Broadcasting large task binary with size 21.0 MiB
26/01/06 17:57:31 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/01/06 17:57:37 WARN DAGScheduler: Broadcasting large task binary with size 27.1 MiB
26/01/06 17:57:45 WARN DAGScheduler: Broadcasting large task binary with size 16.6 MiB
26/01/06 17:57:48 WARN DAGScheduler: Broadcasting large task binary with size 16.6 MiB
26/01/06 17:57:57 WARN DAGScheduler: Broadcasting large task binary with size 1451.0 KiB
26/01/06 17:57:59 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/01/06 17:58:02 WARN DAGScheduler: Broadcasting large task binary with size 3.5 MiB
26/01/06 17:58:06 WARN DAGScheduler: Broadcasting large task binary with size 5.2 MiB
26/01/06 17:58:10 WARN DAGScheduler: Broadcasting large task binary with size 1170.3 KiB
26/01/06 17:58:12 WARN DAGScheduler: Broadca


=== WEATHER MODEL OPTIMIZED ===
{'rmse': 106.32078687636196, 'mse': 11304.109722008781, 'mae': 59.14780823607973, 'r2': 0.2504359389629863}


In [33]:
best_pipe = best_model_weather.bestModel
best_dt = best_pipe.stages[1]

print("\n=== BEST HYPERPARAMETERS (DecisionTreeRegressor) ===")
print("maxDepth:", best_dt.getMaxDepth())
print("maxBins:", best_dt.getMaxBins())
print("minInstancesPerNode:", best_dt.getMinInstancesPerNode())
print("minInfoGain:", best_dt.getMinInfoGain())


=== BEST HYPERPARAMETERS (DecisionTreeRegressor) ===
maxDepth: 10
maxBins: 64
minInstancesPerNode: 3
minInfoGain: 0.0


## FULL MODEL (Ottimizzato)

In [35]:
from pyspark.sql.functions import unix_timestamp, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# ======================================================================
# 1. PREPARAZIONE DATI FULL MODEL
# ======================================================================

num_cols_full = [
    "Distance(mi)", "Severity",
    "Hour", "Weekday", "Month", "Year",
    "Temperature(F)", "Visibility(mi)", "Precipitation(in)", "Wind_Speed(mph)",
    "Humidity(%)",
    "Start_Lat", "Start_Lng"
]

df_reg = df_sample.select(num_cols_full + ["duration_minutes"]).dropna()

# CACHE
df_reg = df_reg.cache()
df_reg.count() 

train_df, test_df = df_reg.randomSplit([0.7, 0.3], seed=42)

# ======================================================================
# 2. PIPELINE
# ======================================================================

assembler = VectorAssembler(inputCols=num_cols_full, outputCol="features")
dt = DecisionTreeRegressor(labelCol="duration_minutes", featuresCol="features")

pipeline = Pipeline(stages=[assembler, dt])

# ======================================================================
# 3. GRID
# ======================================================================

paramGrid = (ParamGridBuilder()
             .addGrid(dt.maxDepth, [10, 20])        # solo 2 valori
             .addGrid(dt.maxBins, [32, 64])         # solo 2 valori
             .addGrid(dt.minInstancesPerNode, [1])  # 1 valore e basta
             .build())

# ======================================================================
# 4. TRAIN-VALIDATION SPLIT
# ======================================================================

evaluator = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="rmse"
)

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    trainRatio=0.8,
    parallelism=2  )

# ======================================================================
# 5. FIT
# ======================================================================

best_model_full = tvs.fit(train_df)

# ======================================================================
# 6. PREDICTION + METRICHE
# ======================================================================

pred = best_model_full.transform(test_df)

evals = {}
for m in ["rmse", "mse", "mae", "r2"]:
    evals[m] = RegressionEvaluator(
        labelCol="duration_minutes", predictionCol="prediction", metricName=m
    ).evaluate(pred)

print("\n=== FULL MODEL OPTIMIZED (FAST VERSION) ===")
print(evals)


26/01/06 18:02:33 WARN DAGScheduler: Broadcasting large task binary with size 1503.1 KiB
26/01/06 18:02:35 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/01/06 18:02:39 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
26/01/06 18:02:45 WARN DAGScheduler: Broadcasting large task binary with size 6.0 MiB
26/01/06 18:02:45 WARN DAGScheduler: Broadcasting large task binary with size 1474.1 KiB
26/01/06 18:02:50 WARN DAGScheduler: Broadcasting large task binary with size 1274.5 KiB
26/01/06 18:02:54 WARN DAGScheduler: Broadcasting large task binary with size 9.1 MiB
26/01/06 18:02:55 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/01/06 18:03:03 WARN DAGScheduler: Broadcasting large task binary with size 1841.9 KiB
26/01/06 18:03:09 WARN DAGScheduler: Broadcasting large task binary with size 13.6 MiB
26/01/06 18:03:10 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB


[1561,475s][warning][gc,alloc] Executor task launch worker for task 11.0 in stage 1279.0 (TID 22932): Retried waiting for GCLocker too often allocating 116243 words
[1561,512s][warning][gc,alloc] Executor task launch worker for task 10.0 in stage 1279.0 (TID 22931): Retried waiting for GCLocker too often allocating 105399 words
[1561,535s][warning][gc,alloc] Executor task launch worker for task 12.0 in stage 1279.0 (TID 22933): Retried waiting for GCLocker too often allocating 67220 words
[1561,555s][warning][gc,alloc] Executor task launch worker for task 15.0 in stage 1279.0 (TID 22936): Retried waiting for GCLocker too often allocating 8591 words
[1561,573s][warning][gc,alloc] Executor task launch worker for task 13.0 in stage 1279.0 (TID 22934): Retried waiting for GCLocker too often allocating 180142 words


26/01/06 18:03:24 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/01/06 18:03:32 WARN DAGScheduler: Broadcasting large task binary with size 19.7 MiB
26/01/06 18:03:34 WARN DAGScheduler: Broadcasting large task binary with size 5.8 MiB


[1592,610s][warning][gc,alloc] Executor task launch worker for task 19.0 in stage 1283.0 (TID 23020): Retried waiting for GCLocker too often allocating 92693 words


26/01/06 18:03:54 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/01/06 18:04:04 WARN DAGScheduler: Broadcasting large task binary with size 1213.8 KiB
26/01/06 18:04:07 WARN DAGScheduler: Broadcasting large task binary with size 15.6 MiB
26/01/06 18:04:12 WARN DAGScheduler: Broadcasting large task binary with size 8.8 MiB
26/01/06 18:04:16 WARN DAGScheduler: Broadcasting large task binary with size 15.6 MiB
26/01/06 18:04:25 WARN DAGScheduler: Broadcasting large task binary with size 1737.0 KiB
26/01/06 18:04:30 WARN DAGScheduler: Broadcasting large task binary with size 12.9 MiB
26/01/06 18:04:43 WARN DAGScheduler: Broadcasting large task binary with size 1962.6 KiB
26/01/06 18:04:48 WARN DAGScheduler: Broadcasting large task binary with size 17.4 MiB
26/01/06 18:05:00 WARN DAGScheduler: Broadcasting large task binary with size 1962.6 KiB
26/01/06 18:05:06 WARN DAGScheduler: Broadcasting large task binary with size 21.6 MiB
26/01/06 18:05:13 WARN DAGScheduler: 


=== FULL MODEL OPTIMIZED (FAST VERSION) ===
{'rmse': 89.78410951561925, 'mse': 8061.186321512711, 'mae': 45.67055319756194, 'r2': 0.4631632928248016}


In [36]:
best_pipe = best_model_full.bestModel
best_dt = best_pipe.stages[1]

print("\n=== BEST HYPERPARAMETERS (DecisionTreeRegressor) ===")
print("maxDepth:", best_dt.getMaxDepth())
print("maxBins:", best_dt.getMaxBins())
print("minInstancesPerNode:", best_dt.getMinInstancesPerNode())
print("minInfoGain:", best_dt.getMinInfoGain())


=== BEST HYPERPARAMETERS (DecisionTreeRegressor) ===
maxDepth: 10
maxBins: 64
minInstancesPerNode: 1
minInfoGain: 0.0


## RANDOM FOREST 

In [8]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

# ======================================================================
# 1. PREPARAZIONE DATI (FULL FEATURES)
# ======================================================================

num_cols_full = [
    "Distance(mi)", "Severity",
    "Hour", "Weekday", "Month", "Year",
    "Temperature(F)", "Visibility(mi)", "Precipitation(in)", "Wind_Speed(mph)",
    "Humidity(%)",
    "Start_Lat", "Start_Lng"
]

df_reg = df_sample.select(num_cols_full + ["duration_minutes"]).dropna()

df_reg = df_reg.cache()
df_reg.count()

train_df, test_df = df_reg.randomSplit([0.7, 0.3], seed=42)

# ======================================================================
# 2. PIPELINE RANDOM FOREST
# ======================================================================

assembler = VectorAssembler(inputCols=num_cols_full, outputCol="features")

rf = RandomForestRegressor(
    labelCol="duration_minutes",
    featuresCol="features",
    seed=42
)

pipeline = Pipeline(stages=[assembler, rf])

# ======================================================================
# 3. GRID 
# ======================================================================

paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [50, 100])
             .addGrid(rf.maxDepth, [8, 12])
             .addGrid(rf.maxBins, [32, 64])
             .addGrid(rf.featureSubsetStrategy, ["sqrt", "log2"])
             .build())

# ======================================================================
# 4. TRAIN-VALIDATION SPLIT
# ======================================================================

evaluator = RegressionEvaluator(
    labelCol="duration_minutes", predictionCol="prediction", metricName="rmse"
)

tvs = TrainValidationSplit(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    trainRatio=0.8,
    parallelism=2
)

# ======================================================================
# 5. FIT
# ======================================================================

tvs_model = tvs.fit(train_df)

# ======================================================================
# 6. PREDICTION + METRICHE
# ======================================================================

best_pipe = tvs_model.bestModel
pred = best_pipe.transform(test_df)

evals = {}
for m in ["rmse", "mse", "mae", "r2"]:
    evals[m] = RegressionEvaluator(
        labelCol="duration_minutes", predictionCol="prediction", metricName=m
    ).evaluate(pred)

print("\n=== RANDOM FOREST OPTIMIZED ===")
print(evals)

# ======================================================================
# 7. PRINT BEST HYPERPARAMETERS
# ======================================================================

best_rf = best_pipe.stages[1]

print("\n=== BEST HYPERPARAMETERS (RandomForestRegressor) ===")
print("numTrees:", best_rf.getNumTrees)
print("maxDepth:", best_rf.getMaxDepth())
print("maxBins:", best_rf.getMaxBins())
print("featureSubsetStrategy:", best_rf.getFeatureSubsetStrategy())

#importanza feature
imp = best_rf.featureImportances
imp_list = [(num_cols_full[i], float(imp[i])) for i in range(len(num_cols_full))]
imp_list = sorted(imp_list, key=lambda x: x[1], reverse=True)

print("\n=== FEATURE IMPORTANCES (RF) ===")
for k, v in imp_list:
    print(f"{k}: {v:.6f}")


26/01/06 18:41:02 WARN BlockManager: Block rdd_56_0 already exists on this machine; not re-adding it
26/01/06 18:42:25 WARN DAGScheduler: Broadcasting large task binary with size 1094.0 KiB
26/01/06 18:42:33 WARN DAGScheduler: Broadcasting large task binary with size 1094.0 KiB
26/01/06 18:42:42 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/01/06 18:42:52 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/01/06 18:44:32 WARN DAGScheduler: Broadcasting large task binary with size 1096.4 KiB
26/01/06 18:44:47 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/01/06 18:44:56 WARN DAGScheduler: Broadcasting large task binary with size 1096.4 KiB
26/01/06 18:45:17 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/01/06 18:46:35 WARN DAGScheduler: Broadcasting large task binary with size 1094.0 KiB
26/01/06 18:46:51 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/01/06 18:47:09 WARN D

[2312,412s][warning][gc,alloc] Executor task launch worker for task 8.0 in stage 381.0 (TID 6856): Retried waiting for GCLocker too often allocating 241187 words


26/01/06 19:23:19 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB
26/01/06 19:23:35 WARN DAGScheduler: Broadcasting large task binary with size 45.1 MiB
26/01/06 19:24:07 WARN DAGScheduler: Broadcasting large task binary with size 1089.7 KiB
26/01/06 19:25:15 WARN DAGScheduler: Broadcasting large task binary with size 11.8 MiB
26/01/06 19:25:27 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB
26/01/06 19:25:29 WARN DAGScheduler: Broadcasting large task binary with size 5.3 MiB
26/01/06 19:25:48 WARN DAGScheduler: Broadcasting large task binary with size 1250.2 KiB
26/01/06 19:25:51 WARN DAGScheduler: Broadcasting large task binary with size 4.0 MiB
26/01/06 19:26:38 WARN DAGScheduler: Broadcasting large task binary with size 1193.0 KiB
26/01/06 19:26:43 WARN DAGScheduler: Broadcasting large task binary with size 7.6 MiB
26/01/06 19:27:40 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/01/06 19:27:50 WARN DAGScheduler: Broadc

[3692,230s][warning][gc,alloc] Executor task launch worker for task 11.0 in stage 471.0 (TID 8559): Retried waiting for GCLocker too often allocating 65410 words
[3692,233s][warning][gc,alloc] Executor task launch worker for task 10.0 in stage 471.0 (TID 8558): Retried waiting for GCLocker too often allocating 190882 words


[3710,210s][warning][gc,alloc] Executor task launch worker for task 11.0 in stage 471.0 (TID 8559): Retried waiting for GCLocker too often allocating 34868 words
[3710,217s][warning][gc,alloc] Executor task launch worker for task 18.0 in stage 471.0 (TID 8566): Retried waiting for GCLocker too often allocating 225618 words


26/01/06 19:46:11 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB
26/01/06 19:46:18 WARN DAGScheduler: Broadcasting large task binary with size 24.0 MiB
26/01/06 19:46:53 WARN DAGScheduler: Broadcasting large task binary with size 6.1 MiB
26/01/06 19:47:00 WARN DAGScheduler: Broadcasting large task binary with size 21.8 MiB
26/01/06 19:47:29 WARN DAGScheduler: Broadcasting large task binary with size 5.7 MiB
26/01/06 19:47:36 WARN DAGScheduler: Broadcasting large task binary with size 8.8 MiB
26/01/06 19:47:46 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB



=== RANDOM FOREST OPTIMIZED ===
{'rmse': 89.0581086230283, 'mse': 7931.346711511107, 'mae': 46.881750191054, 'r2': 0.4718099939323246}

=== BEST HYPERPARAMETERS (RandomForestRegressor) ===
numTrees: 100
maxDepth: 12
maxBins: 64
featureSubsetStrategy: sqrt

=== FEATURE IMPORTANCES (RF) ===
Distance(mi): 0.404267
Year: 0.287767
Hour: 0.067521
Month: 0.065567
Start_Lat: 0.049967
Severity: 0.044419
Start_Lng: 0.029417
Temperature(F): 0.014185
Precipitation(in): 0.013248
Humidity(%): 0.009316
Wind_Speed(mph): 0.007506
Weekday: 0.004736
Visibility(mi): 0.002083


## DT CON FEATURE CAT

In [25]:
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

label_col = "duration_minutes"

cat_cols = ["weather_grouped", "State", "Wind_Direction"]

num_cols = [
    "Month", "Humidity(%)", "Precipitation(in)", "Temperature(F)",
    "Visibility(mi)", "Wind_Speed(mph)", "Distance(mi)",
    "Year", "Day", "Weekday", "Hour", "Sunrise_Sunset"
]

bool_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction",
    "No_Exit", "Railway", "Roundabout", "Station", "Stop",
    "Traffic_Calming", "Traffic_Signal", "Turning_Loop"
]

# =========================
# PREPROCESSING
# =========================
df1 = df.filter(
    (col(label_col) > 0) & (col(label_col) < 1440)
)

for c in bool_cols:
    if c in df1.columns:
        df1 = df1.withColumn(c, col(c).cast("int"))

used_cols = (
    cat_cols +
    num_cols +
    bool_cols +
    [label_col]
)

df1 = df1.select(*used_cols).dropna(subset=[label_col])
df1 = df1.sample(fraction=0.4, seed=42)

# =========================
# PIPELINE
# =========================
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

feature_cols = (
    num_cols +
    bool_cols +
    [f"{c}_idx" for c in cat_cols]
)

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

dt = DecisionTreeRegressor(
    labelCol=label_col,
    featuresCol="features",
    seed=42
)

pipeline_dt = Pipeline(stages=indexers + [assembler, dt])

train_df, test_df = df1.randomSplit([0.7, 0.3], seed=42)

# =========================
# TUNING
# =========================
paramGrid_dt = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10, 15])
    .addGrid(dt.maxBins, [64, 128])
    .addGrid(dt.minInstancesPerNode, [1, 5])
    .addGrid(dt.minInfoGain, [0.0, 0.05])
    .build()
)

rmse_eval = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="rmse"
)

tvs_dt = TrainValidationSplit(
    estimator=pipeline_dt,
    estimatorParamMaps=paramGrid_dt,
    evaluator=rmse_eval,
    trainRatio=0.8,
    seed=42,
    parallelism=4
)

dt_model = tvs_dt.fit(train_df).bestModel
dt_pred = dt_model.transform(test_df)

dt_best = dt_model.stages[-1]

print("Decision Tree – Best Parameters")
print("maxDepth:", dt_best.getMaxDepth())
print("maxBins:", dt_best.getMaxBins())
print("minInstancesPerNode:", dt_best.getMinInstancesPerNode())
print("minInfoGain:", dt_best.getMinInfoGain())

print("DT RMSE:", rmse_eval.evaluate(dt_pred))
print("DT MAE :", RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mae").evaluate(dt_pred))
print("DT MSE :", RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mse").evaluate(dt_pred))
print("DT R2  :", RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2").evaluate(dt_pred))


26/01/10 19:19:02 WARN DAGScheduler: Broadcasting large task binary with size 1169.2 KiB
26/01/10 19:19:05 WARN DAGScheduler: Broadcasting large task binary with size 1863.7 KiB
26/01/10 19:19:21 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
26/01/10 19:19:37 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/01/10 19:19:38 WARN DAGScheduler: Broadcasting large task binary with size 1148.0 KiB
26/01/10 19:19:56 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/01/10 19:19:58 WARN DAGScheduler: Broadcasting large task binary with size 1814.4 KiB
26/01/10 19:20:05 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/01/10 19:20:14 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/01/10 19:20:15 WARN DAGScheduler: Broadcasting large task binary with size 1091.8 KiB
26/01/10 19:20:18 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/01/10 19:20:21 WARN DAGScheduler: Br

Decision Tree – Best Parameters
maxDepth: 15
maxBins: 128
minInstancesPerNode: 5
minInfoGain: 0.05


26/01/10 19:31:22 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/01/10 19:31:39 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB


DT RMSE: 85.5825121160413


26/01/10 19:31:39 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/01/10 19:31:58 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB


...

26/01/10 19:31:58 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/01/10 19:32:12 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB


DT MSE : 7324.366380092357


26/01/10 19:32:13 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB


DT R2  : 0.5084243214355554


26/01/10 19:32:30 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB


## RANDOM FOREST CON FEATURE CAT

In [7]:
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

label_col = "duration_minutes"

cat_cols = ["weather_grouped", "State", "Wind_Direction"]

num_cols = [
    "Month", "Humidity(%)", "Precipitation(in)", "Temperature(F)",
    "Visibility(mi)", "Wind_Speed(mph)", "Distance(mi)",
    "Year", "Day", "Weekday", "Hour", "Sunrise_Sunset"
]

bool_cols = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction",
    "No_Exit", "Railway", "Roundabout", "Station", "Stop",
    "Traffic_Calming", "Traffic_Signal", "Turning_Loop"
]

# =========================
# PREPROCESSING
# =========================
df_rf = df

for c in bool_cols:
    if c in df_rf.columns:
        df_rf = df_rf.withColumn(c, col(c).cast("int"))

used_cols = cat_cols + num_cols + bool_cols + [label_col]
df_rf = df_rf.select(*used_cols).dropna(subset=[label_col])
df_rf = df_rf.sample(fraction=0.4, seed=42)

# =========================
# STRING INDEXER
# =========================
indexers_rf = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_idx",
        handleInvalid="keep"
    )
    for c in cat_cols
]

# =========================
# VECTOR ASSEMBLER
# =========================
feature_cols_rf = (
    num_cols +
    bool_cols +
    [f"{c}_idx" for c in cat_cols]
)

assembler_rf = VectorAssembler(
    inputCols=feature_cols_rf,
    outputCol="features",
    handleInvalid="keep"
)

# =========================
# RANDOM FOREST
# =========================
rf = RandomForestRegressor(
    labelCol=label_col,
    featuresCol="features",
    seed=42
)

pipeline_rf = Pipeline(stages=indexers_rf + [assembler_rf, rf])

# =========================
# TRAIN / TEST
# =========================
train_rf, test_rf = df_rf.randomSplit([0.7, 0.3], seed=42)

# =========================
# TUNING
# =========================
paramGrid_rf = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [30, 40])
    .addGrid(rf.maxDepth, [10, 12])
    .addGrid(rf.maxBins, [64])
    .addGrid(rf.minInstancesPerNode, [5])
    .build()
)


rmse_eval = RegressionEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="rmse"
)

tvs_rf = TrainValidationSplit(
    estimator=pipeline_rf,
    estimatorParamMaps=paramGrid_rf,
    evaluator=rmse_eval,
    trainRatio=0.8,
    seed=42,
    parallelism=4
)

rf_model = tvs_rf.fit(train_rf).bestModel
rf_pred = rf_model.transform(test_rf)

# =========================
# RISULTATI
# =========================
rf_best = rf_model.stages[-1]

print("Random Forest – Best Parameters")
print("numTrees:", rf_best.getNumTrees)
print("maxDepth:", rf_best.getMaxDepth())
print("maxBins:", rf_best.getMaxBins())
print("minInstancesPerNode:", rf_best.getMinInstancesPerNode())

print("RF RMSE:", rmse_eval.evaluate(rf_pred))
print("RF MAE :", RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mae").evaluate(rf_pred))
print("RF MSE :", RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="mse").evaluate(rf_pred))
print("RF R2  :", RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2").evaluate(rf_pred))


26/01/10 21:41:28 WARN DAGScheduler: Broadcasting large task binary with size 1218.3 KiB
26/01/10 21:41:38 WARN DAGScheduler: Broadcasting large task binary with size 1218.4 KiB
26/01/10 21:41:46 WARN DAGScheduler: Broadcasting large task binary with size 1727.5 KiB
26/01/10 21:41:56 WARN DAGScheduler: Broadcasting large task binary with size 1727.5 KiB
26/01/10 21:42:07 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/01/10 21:42:21 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/01/10 21:42:33 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/01/10 21:42:46 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/01/10 21:43:02 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/01/10 21:43:20 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
26/01/10 21:43:35 WARN DAGScheduler: Broadcasting large task binary with size 5.7 MiB
26/01/10 21:43:51 WARN DAGScheduler: Broad

Random Forest – Best Parameters
numTrees: 40
maxDepth: 12
maxBins: 64
minInstancesPerNode: 5


RF RMSE: 87.24338752439097


RF MAE : 45.778883593298424


RF MSE : 7611.408666731057


RF R2  : 0.4891594458806371
